In [1]:
from datetime import date

import pandas as pd
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
patstat = PatstatClient()          # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()                 # SQLAlchemy session used by all ORM queries below

This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [3]:
CANDIDATE_DRONE_KEYWORDS = (
    # General UAV terminology
    "unmanned aerial vehicle",
    "unmanned aerial system",
    "unmanned aircraft",
    "uncrewed aerial vehicle",
    "uncrewed aircraft",
    "unmanned air vehicle",
    "remotely piloted aircraft",
    "remotely piloted aerial vehicle",
    "remotely controlled aircraft",
    "unmanned flying vehicle",

    # Common acronyms
    "UAV",
    "RPAS",

    # Drone types
    "quadcopter",
    "quadrotor",
    "quad-rotor",
    "hexacopter",
    "hexarotor",
    "octocopter",

    # Additional UAV terminology
    "aerial drone",
    "flying drone",
    "drone aircraft",
    "fixed-wing drone",
    "flying robot",
    "micro air vehicle",
)


In [4]:
#"remote-controlled aircraft","remote controlled aircraft",

DRONE_KEYWORDS = (
    # "unmanned/unpiloted" + aircraft/aerial (both papers)
    "unmanned aerial", "unmanned aircraft", "unmanned air vehicle",
    "unpiloted aerial", "unpiloted aircraft", "unpiloted air vehicle",
    # remote* within 1 word of aircraft (DTI)
    "remotely piloted aircraft",
    "micro air vehicle",                                    # DTI
    # rotor types (Gorry)
    "quadrotor", "quadrocopter", "quadcopter", "multicopter", "multirotor", "multi-rotor",
    "hexacopter", "octocopter",
    " uav", "uav ",
    "drone",                                                # contested: keep only if hand-labelling supports it
)
#these are  possible keywords found in scientific paper

In [5]:
DRONE_IPC_CODES = (
    "B64U",
)

In [6]:
DRONE_CPC_CODES = (
    "B64U",
    "B64C39/024",
)

In [7]:
def keyword_condition(terms):
    """OR-condition: any term appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Application-level fields retrieved by the search queries.
# These fields describe each application; they do not change the search criteria.
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,              # Unique application identifier
    TLS201_APPLN.docdb_family_id,       # Links applications belonging to the same DOCDB family
    TLS201_APPLN.appln_auth,            # Filing authority or patent office
    TLS201_APPLN.appln_nr,              # Application number
    TLS201_APPLN.appln_kind,            # Application kind code
    TLS201_APPLN.appln_filing_date,     # Filing date of this application
    TLS201_APPLN.earliest_filing_year,  # Earliest filing year associated with this application
    TLS201_APPLN.granted,               # Grant indicator: "Y" means granted
)


def query_by_keywords(*columns):
    """Applications whose title or abstract contains an Drone keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )

In [8]:
#KEYWORD SEARCH

q_keywords = query_by_keywords(
    *APPLN_COLUMNS,
    TLS202_APPLN_TITLE.appln_title,
    TLS203_APPLN_ABSTR.appln_abstract,
).distinct()

df_keywords = patstat.df(q_keywords)
df_keywords.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,14971374,36650390,DE,102005002278,A,2005-01-18,2005,N,Method of monitoring land or terrain and deter...,The method involves the aircraft (10) flying a...
1,473308104,57589963,US,201314040040,A,2013-09-27,2012,Y,Unmanned aircraft system (UAS) with active ene...,A method of harvesting and managing energy fro...
2,422949556,51729633,US,201414204634,A,2014-03-11,2013,Y,Unmanned aerial vehicle and methods for contro...,One variation of a method for imaging an area ...
3,457561603,56164630,US,201414282419,A,2014-05-20,2014,Y,Secure payload deliveries via unmanned aerial ...,A device receives a request for a flight path ...
4,572681286,63761662,US,202016838417,A,2020-04-02,2014,Y,Accident reconstruction implementing unmanned ...,Unmanned aerial vehicles (UAVs) may facilitate...
5,419503029,50980760,GB,201408097,A,2014-05-07,2014,N,Method of using a device capable of controlled...,"A device 40 such as a UAV, drone or quad-copte..."
6,418088109,50708445,CN,201410084493,A,2014-03-07,2014,Y,Centralized monitoring system for power transm...,The invention discloses a centralized monitori...
7,479339018,58798329,US,201514958408,A,2015-12-03,2015,Y,Drone piggybacking on vehicles,A system for providing drone piggybacking on v...
8,472359922,54600750,EP,16803675,A,2016-05-26,2015,N,FIRE EXTINGUISHING FIREFIGHTING DRONE,The present invention relates to a fire exting...
9,471673748,57289563,CN,201610579453,A,2016-07-21,2015,Y,Multi-rotor unmanned aerial vehicle detection ...,The invention discloses a multi-rotor unmanned...


In [9]:
#IPC SEARCH
q_ipc = query_by_ipc(*APPLN_COLUMNS, TLS209_APPLN_IPC.ipc_class_symbol).distinct()

df_ipc = patstat.df(q_ipc)
df_ipc.head()

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,ipc_class_symbol
0,513740868,48290734,US,201916265314,A,2019-02-01,2011,Y,B64U 10/25
1,513740868,48290734,US,201916265314,A,2019-02-01,2011,Y,B64U 20/10
2,556168280,49580728,US,202117233087,A,2021-04-16,2012,Y,B64U 101/00
3,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 10/13
4,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 80/60


In [10]:
#CPC SEARCH
q_cpc = query_by_cpc(*APPLN_COLUMNS, TLS224_APPLN_CPC.cpc_class_symbol).distinct()

df_cpc = patstat.df(q_cpc)
df_cpc.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,cpc_class_symbol
0,52104346,24390749,US,59725632,A,1932-03-07,1932,Y,B64U 10/60
1,54372193,25514354,US,96850378,A,1978-12-11,1978,Y,B64U 10/60
2,46566715,22019089,US,5881587,A,1987-06-05,1987,Y,B64U 70/30
3,50237261,27320303,US,36900589,A,1989-06-20,1988,Y,B64U 10/60
4,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64C 39/024
5,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 10/10
6,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 10/25
7,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/20
8,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/21
9,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/294


In [11]:
# COMBINE THE SEARCH RESULTS

# Select application IDs found by at least one search strategy:
# keywords OR IPC codes OR CPC codes.
# UNION removes duplicate IDs across the three searches.
appln_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(appln_id)
    .union(
        query_by_ipc(appln_id),
        query_by_cpc(appln_id),
    )
    .subquery()
)

# Retrieve application details, titles and abstracts for the selected IDs.
# Outer joins preserve applications even when a title or abstract is missing.
q_drone = (
    db.query(
        *APPLN_COLUMNS,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
    )
    .select_from(TLS201_APPLN)
    .join(
        drone_ids,
        TLS201_APPLN.appln_id == drone_ids.c.appln_id,
    )
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
    )
    .outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
    )
    .distinct()  # Remove identical result rows
    .order_by(TLS201_APPLN.appln_id)
)

# Execute the query and store the results in a pandas DataFrame.
# This is still an application-level dataset, not a family-level dataset.
df_drone = patstat.df(q_drone)

df_drone.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,24746,37809459,EP,06802616,A,2006-08-30,2005,N,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,"Co-axial, multi-rotor wind turbines, producing..."
1,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...
2,352060,33524069,EP,04017945,A,2004-07-29,2003,N,A flying machine,The flying machine (1) comprises a supporting ...
3,2755148,37809459,AU,2006284845,A,2006-08-30,2005,Y,Multi-rotor wind turbine supported by continuo...,"Co-axial, multi-rotor wind turbines, producing..."
4,4305581,25672538,CA,594383,A,1989-03-21,1987,Y,MONITORING SYSTEM FOR POWER LINES AND RIGHT-OF...,Application Of: Roosevelt A. Fernandes For: Mo...
5,5009380,37809459,CA,2620922,A,2006-08-30,2005,N,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,"Co-axial, multi-rotor wind turbines, producing..."
6,6750614,26877844,CN,01807474,A,2001-02-12,2000,N,Remotely piloted aircraft,None
7,7611249,5221267,CN,98111274,A,1998-05-04,1998,N,Wind motor,"The present invention relates to a wind motor,..."
8,13609024,7942894,DE,20010668,U,2000-06-15,2000,Y,Windenergieanlage Multi-Rotor-Prinzip,None
9,14034049,11074674,DE,60119462,T,2001-09-10,2000,Y,Unbemannte mobile Vorrichtung,None


In [12]:
# How much does each strategy contribute? (counts of distinct applications)
pd.Series(
    {
        "Keywords": df_keywords["appln_id"].nunique(),
        "IPC": df_ipc["appln_id"].nunique(),
        "CPC": df_cpc["appln_id"].nunique(),
        "Total union": df_drone["appln_id"].nunique(),
    },
    name="Applications"
)

Keywords       2201
IPC             126
CPC            2190
Total union    3467
Name: Applications, dtype: int64

In [13]:
# INITIAL DATA QUALITY CHECKS
# These checks report potential issues without modifying or deleting any data.

# Compare the number of rows with the number of distinct applications.
# A difference indicates that some application IDs appear on multiple rows.
print("Total rows:", len(df_drone))
print("Distinct applications:", df_drone["appln_id"].nunique())

# A usable DOCDB family ID must be present and greater than zero.
valid_family = (
    df_drone["docdb_family_id"].notna()
    & (df_drone["docdb_family_id"] > 0)
)

print(
    "Distinct families with a valid ID:",
    df_drone.loc[valid_family, "docdb_family_id"].nunique(),
)

print(
    "Applications without a valid family ID:",
    df_drone.loc[~valid_family, "appln_id"].nunique(),
)

# Convert years to numeric values for consistent missing-value checks.
# PATSTAT uses 9999 for an unavailable year.
priority_year = pd.to_numeric(
    df_drone["earliest_filing_year"],
    errors="coerce",
)

missing_priority_year = (
    priority_year.isna()
    | (priority_year == 9999)
)

print(
    "Applications with an unavailable earliest filing year:",
    df_drone.loc[missing_priority_year, "appln_id"].nunique(),
)

Total rows: 3467
Distinct applications: 3467
Distinct families with a valid ID: 2470
Applications without a valid family ID: 0
Applications with an unavailable earliest filing year: 0


In [14]:
# IDENTIFY THE SEARCH STRATEGIES THAT RETRIEVED EACH APPLICATION
# Create a diagnostic copy without changing the original dataset.
from IPython.display import display

df_diagnostics = df_drone.copy()

# A value of True means that the application was found by that strategy.
df_diagnostics["found_by_keywords"] = (
    df_diagnostics["appln_id"].isin(df_keywords["appln_id"])
)

df_diagnostics["found_by_ipc"] = (
    df_diagnostics["appln_id"].isin(df_ipc["appln_id"])
)

df_diagnostics["found_by_cpc"] = (
    df_diagnostics["appln_id"].isin(df_cpc["appln_id"])
)


# Combine the flags into a readable label.
def get_retrieval_strategy(row):
    strategies = []

    if row["found_by_keywords"]:
        strategies.append("Keywords")
    if row["found_by_ipc"]:
        strategies.append("IPC")
    if row["found_by_cpc"]:
        strategies.append("CPC")

    return " + ".join(strategies)


df_diagnostics["retrieval_strategy"] = df_diagnostics.apply(
    get_retrieval_strategy,
    axis=1,
)

# Count distinct applications for each combination.
# Each application is counted in exactly one combination.
strategy_summary = (
    df_diagnostics
    .groupby("retrieval_strategy")["appln_id"]
    .nunique()
    .rename("n_applications")
    .sort_values(ascending=False)
    .reset_index()
)

display(strategy_summary)

# Show applications retrieved only by keywords.
# These are candidates for inspection, not automatically irrelevant results.
keyword_only = df_diagnostics.loc[
    df_diagnostics["found_by_keywords"]
    & ~df_diagnostics["found_by_ipc"]
    & ~df_diagnostics["found_by_cpc"]
].copy()

print(
    "Applications retrieved only by keywords:",
    keyword_only["appln_id"].nunique(),
)

display(
    keyword_only[
        ["appln_id", "appln_title", "retrieval_strategy"]
    ].head(20)
)

,retrieval_strategy,n_applications
0,Keywords,1244
1,CPC,1213
2,Keywords + CPC,884
3,Keywords + IPC + CPC,57
4,IPC + CPC,36
5,IPC,17
6,Keywords + IPC,16


Applications retrieved only by keywords: 1244


,appln_id,appln_title,retrieval_strategy
0,24746,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,Keywords
3,2755148,Multi-rotor wind turbine supported by continuo...,Keywords
4,4305581,MONITORING SYSTEM FOR POWER LINES AND RIGHT-OF...,Keywords
5,5009380,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,Keywords
7,7611249,Wind motor,Keywords
8,13609024,Windenergieanlage Multi-Rotor-Prinzip,Keywords
11,14971374,Method of monitoring land or terrain and deter...,Keywords
12,16419618,Enhanced passive coherent location techniques ...,Keywords
13,19115737,Hydrodynamic or streamlined profile for formin...,Keywords
14,21513231,Multi-rotor wind turbine,Keywords


In [ ]:
# INSPECT AN APPLICATION AND ITS RETRIEVAL EVIDENCE
def inspect_application(application_id):
    rows = df_diagnostics.loc[
        df_diagnostics["appln_id"] == application_id
    ]

    if rows.empty:
        print("Application not found in the retrieved dataset.")
        return

    # Retrieve the classification symbols that matched the search queries.
    # These are not necessarily all the codes assigned to the application.
    ipc_codes = sorted(
        df_ipc.loc[
            df_ipc["appln_id"] == application_id,
            "ipc_class_symbol",
        ].dropna().unique().tolist()
    )

    cpc_codes = sorted(
        df_cpc.loc[
            df_cpc["appln_id"] == application_id,
            "cpc_class_symbol",
        ].dropna().unique().tolist()
    )

    # Inspect every text record if the application appears on multiple rows.
    for _, row in rows.iterrows():
        title = (
            str(row["appln_title"])
            if pd.notna(row["appln_title"])
            else ""
        )
        abstract = (
            str(row["appln_abstract"])
            if pd.notna(row["appln_abstract"])
            else ""
        )

        # Reproduce the current case-insensitive substring matching.
        # This also reveals matches occurring inside longer words.
        title_matches = sorted({
            term for term in DRONE_KEYWORDS
            if term.lower() in title.lower()
        })

        abstract_matches = sorted({
            term for term in DRONE_KEYWORDS
            if term.lower() in abstract.lower()
        })

        print("Application ID:", application_id)
        print("DOCDB family ID:", row["docdb_family_id"])
        print("Retrieved by:", row["retrieval_strategy"])
        print("Keywords in title:", title_matches)
        print("Keywords in abstract:", abstract_matches)
        print("Matched IPC codes:", ipc_codes)
        print("Matched CPC codes:", cpc_codes)
        print("\nTITLE:\n", title or "[Not available]")
        print("\nABSTRACT:\n", abstract or "[Not available]")
        print()


# Example: inspect the wind-turbine application shown in the saved results.
#inspect_application(51564414)

In [16]:
# GROUP THE RETRIEVED APPLICATIONS BY DOCDB FAMILY

# Keep applications with a valid family identifier.
valid_family = (
    df_diagnostics["docdb_family_id"].notna()
    & (df_diagnostics["docdb_family_id"] > 0)
)

df_family_hits = df_diagnostics.loc[valid_family].copy()

# Create one row per family.
# These counts refer only to retrieved applications, not all family members.
df_families = (
    df_family_hits
    .groupby("docdb_family_id", as_index=False)
    .agg(
        n_matched_applications=("appln_id", "nunique"),
        matched_application_ids=(
            "appln_id",
            lambda ids: sorted(set(ids)),
        ),
        found_by_keywords=("found_by_keywords", "any"),
        found_by_ipc=("found_by_ipc", "any"),
        found_by_cpc=("found_by_cpc", "any"),
    )
    .sort_values("docdb_family_id")
    .reset_index(drop=True)
)

# Reuse the function defined in the diagnostic cell.
# At family level, different members may match different strategies.
df_families["retrieval_strategy"] = df_families.apply(
    get_retrieval_strategy,
    axis=1,
)

print(
    "Retrieved applications with a valid family ID:",
    df_family_hits["appln_id"].nunique(),
)
print("Distinct DOCDB families:", len(df_families))

display(df_families.head(10))

Retrieved applications with a valid family ID: 3467
Distinct DOCDB families: 2470


,docdb_family_id,n_matched_applications,matched_application_ids,found_by_keywords,found_by_ipc,found_by_cpc,retrieval_strategy
0,5221267,1,[7611249],True,False,False,Keywords
1,7942894,1,[13609024],True,False,False,Keywords
2,11074674,2,"[14034049, 54342525]",False,False,True,CPC
3,11459538,1,[52322637],False,False,True,CPC
4,19895696,1,[42023759],True,False,False,Keywords
5,19914776,2,"[42123841, 53662805]",False,False,True,CPC
6,22019089,1,[46566715],False,False,True,CPC
7,23142817,1,[49394130],True,False,True,Keywords + CPC
8,23143245,1,[49394876],True,False,True,Keywords + CPC
9,24390749,1,[52104346],False,False,True,CPC


In [17]:
# SELECT A REPRODUCIBLE RANDOM SAMPLE OF FAMILIES (100)

validation_sample = (
    df_families
    .sample(
        n=min(100, len(df_families)),
        random_state=42,
    )
    .reset_index(drop=True)
)

# Fields to complete during manual validation.
# Suggested relevance labels: relevant, not relevant, uncertain.
validation_sample["relevance"] = ""
validation_sample["reason"] = ""

print("Families selected for validation:", len(validation_sample))

display(
    validation_sample[
        [
            "docdb_family_id",
            "n_matched_applications",
            "retrieval_strategy",
            "relevance",
            "reason",
        ]
    ].head(10)
)

Families selected for validation: 100


,docdb_family_id,n_matched_applications,retrieval_strategy,relevance,reason
0,62909257,1,Keywords + CPC,,
1,80738296,1,CPC,,
2,72423711,1,Keywords + CPC,,
3,69597174,1,Keywords + CPC,,
4,54198982,3,Keywords + CPC,,
5,62567684,4,CPC,,
6,55589040,1,Keywords + CPC,,
7,72075337,1,Keywords,,
8,51422096,1,Keywords + CPC,,
9,76329056,1,CPC,,


In [18]:
# INSPECT THE RETRIEVED APPLICATIONS OF ONE FAMILY

def inspect_family(family_id):
    applications = (
        df_family_hits.loc[
            df_family_hits["docdb_family_id"] == family_id,
            "appln_id",
        ]
        .drop_duplicates()
        .sort_values()
    )

    if applications.empty:
        print("Family not found in the retrieved dataset.")
        return

    print("DOCDB family:", family_id)
    print("Retrieved applications:", len(applications))
    print()

    # Reuse the application inspection function.
    # It prints full texts and the evidence explaining each match.
    for application_id in applications:
        inspect_application(application_id)


# Inspect the first family in the random sample.
if not validation_sample.empty:
    inspect_family(
        validation_sample.iloc[0]["docdb_family_id"]
    )

DOCDB family: 62909257
Retrieved applications: 1

Application ID: 523028742
DOCDB family ID: 62909257
Retrieved by: Keywords + CPC
Keywords in title: ['unmanned aerial']
Keywords in abstract: [' uav', 'uav ', 'unmanned aerial']
Matched IPC codes: []
Matched CPC codes: ['B64C  39/024', 'B64U  10/13', 'B64U  30/20', 'B64U2101/00', 'B64U2101/28', 'B64U2201/10', 'B64U2201/104', 'B64U2201/20']

TITLE:
 Unmanned aerial vehicle for painting structures

ABSTRACT:
 An unmanned aerial vehicle (UAV) includes a sprayer configured to generate a pressurized fluid flow and a nozzle configured to receive the pressurized fluid from the sprayer and to generate a spray fan to apply the fluid to a surface. The UAV includes sensors and a control unit to control both flight of the UAV and spraying by the sprayer. The fluid can be stored onboard the UAV in a reservoir or can be remotely stored and pumped to the UAV. The UAV control unit can be preloaded with a spray plan and a flight plan, or the UAV can be 